# L07 – Regularisering

**Tid:** ca 2 h · **Del B: Träna ett nätverk väl**

## Mål
1. Använda **early stopping** och förstå varför det nästan alltid ska vara med.
2. Förstå **weight decay** och koppla det till Ridge och `C` i sklearn.
3. Förstå **dropout** och skillnaden mellan `model.train()` och `model.eval()`.
4. Förstå grundidén med **batch norm**.
5. Veta vilken teknik du tar till **när**.

**Var i arbetsflödet?** Steg 8. Diagnosen i steg 7 visade *överanpassning*: train blir mycket bättre än val. Regularisering gör det svårare för modellen att lära sig träningsdatan utantill.

> Kräver `dlkit` från L05. I den här lektionen bygger vi ut `dlkit.fit` med early stopping.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.plots import plot_history

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()

## 1. Utgångsläget: en modell som överanpassar

Precis som i L06 tränar vi en stor MLP på bara **2 000** bilder. Mindre data gör överanpassningen tydlig och träningen snabb. Det är samma fenomen som uppstår med stora modeller på stora dataset, bara i liten skala.

In [ ]:
train_loader, val_loader, _ = fashion_mnist_loaders(train_subset=2000)

from dlkit.train import fit   # versionen från L05

def make_mlp(hidden=512):
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(),
                         nn.Linear(hidden, 10))

set_seed(42)
model = make_mlp().to(device)
h_base = fit(model, train_loader, val_loader, loss_fn,
             torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=40, verbose=False)
plot_history(h_base, "Ingen regularisering")
results = {"ingen": h_base}

## 2. Early stopping

**Idé:** följ val loss under träningen och **behåll vikterna från den bästa epoken**. Avbryt när val loss inte har förbättrats på `patience` epoker.

- **Varför fungerar det?** I början lär sig modellen generella mönster, och båda kurvorna sjunker. Senare börjar den lära sig brus i träningsdatan, och då stiger val loss. Early stopping stannar vid brytpunkten.
- **sklearn:** `MLPClassifier(early_stopping=True, n_iter_no_change=10)` och `HistGradientBoostingClassifier(early_stopping=True)` gör samma sak.
- **`patience`:** för litet (1–2) och du stannar på grund av tillfälligt brus. För stort och du slösar tid. Med 5–10 epoker brukar det bli bra.

Vi bygger in det i `dlkit.fit`. Cellen nedan skriver över `dlkit/train.py` med en ny version. Det nya är markerat med `# NEW (L07)`. Allt annat är som förut, så gamla anrop fungerar fortfarande.

In [ ]:
%%writefile ../dlkit/train.py
import copy
import time

import torch


def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """One pass over the training data. Returns (mean loss, accuracy)."""
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def evaluate(model, loader, loss_fn, device):
    """Evaluate without gradients. Returns (mean loss, accuracy)."""
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    for X, y in loader:
        X, y = X.to(device), y.to(device)
        logits = model(X)
        total_loss += loss_fn(logits, y).item() * len(y)
        correct += (logits.argmax(dim=1) == y).sum().item()
        n += len(y)
    return total_loss / n, correct / n


@torch.no_grad()
def predict(model, loader, device):
    """Return (y_true, y_pred, probs) as NumPy arrays – ready for sklearn.metrics."""
    model.eval()
    ys, probs = [], []
    for X, y in loader:
        logits = model(X.to(device))
        probs.append(torch.softmax(logits, dim=1).cpu())
        ys.append(y)
    probs = torch.cat(probs)
    return torch.cat(ys).numpy(), probs.argmax(dim=1).numpy(), probs.numpy()


def fit(model, train_loader, val_loader, loss_fn, optimizer, device, epochs=10,
        patience=None, verbose=True):
    """Train for up to `epochs` epochs. Returns a history dict.

    patience: stop when val loss has not improved for this many epochs, and
              restore the weights from the best epoch (early stopping).   # NEW (L07)
    """
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0  # NEW (L07)
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        va_loss, va_acc = evaluate(model, val_loader, loss_fn, device)
        for key, value in zip(history, (tr_loss, tr_acc, va_loss, va_acc)):
            history[key].append(value)

        # NEW (L07): remember the best weights seen so far
        improved = va_loss < best_loss
        if improved:
            best_loss, best_epoch, bad_epochs = va_loss, epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1

        if verbose:
            print(f"epok {epoch:3d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f} | {time.time() - t0:.1f}s"
                  + ("  *" if improved else ""))

        if patience is not None and bad_epochs >= patience:  # NEW (L07)
            if verbose:
                print(f"Early stopping: ingen förbättring på {patience} epoker.")
            break

    if patience is not None and best_state is not None:  # NEW (L07)
        model.load_state_dict(best_state)
        if verbose:
            print(f"Återställde vikterna från epok {best_epoch} (val loss {best_loss:.4f}).")
    history["best_epoch"] = best_epoch
    return history

In [ ]:
import importlib, dlkit.train
importlib.reload(dlkit.train)          # ladda om den nya versionen i den här kerneln
from dlkit.train import fit, evaluate

set_seed(42)
model = make_mlp().to(device)
h_es = fit(model, train_loader, val_loader, loss_fn,
           torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=40, patience=5)
print("val acc efter återställning:", round(evaluate(model, val_loader, loss_fn, device)[1], 3))

`state_dict()` är modellens vikter, en ordbok med tensorer. Vi går igenom den i L09. `copy.deepcopy` behövs för att spara en *kopia*, eftersom vikterna annars fortsätter att ändras.

Early stopping är billigt och har nästan bara fördelar. **Använd det i princip alltid.** Det kombineras med teknikerna nedan.

## 3. Weight decay (L2-regularisering)

**Idé:** straffa stora vikter. Varje steg krymper alla vikter lite mot 0. Modellen kan då inte förlita sig på enstaka extrema vikter, och funktionen den lär sig blir "slätare".

**Koppling till sklearn:**
- `Ridge(alpha=...)` lägger till $\alpha \lVert w \rVert^2$ i lossen, vilket är exakt L2-regularisering.
- `LogisticRegression(C=...)` har $C = 1/\alpha$. **Litet `C` ger stark regularisering.**
- I PyTorch anges styrkan direkt i optimeraren: `weight_decay=...`.

**AdamW i stället för Adam:** med Adam samverkar weight decay dåligt med de anpassade steglängderna. `AdamW` gör det på rätt sätt och är standardvalet när du vill ha weight decay. Typiska värden är 0,01–0,1.

In [ ]:
for wd in [0.01, 0.1, 1.0]:
    set_seed(42)
    model = make_mlp().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    results[f"weight decay {wd}"] = fit(model, train_loader, val_loader, loss_fn, opt, device,
                                        epochs=40, verbose=False)

def summarize(results):
    print(f"{'variant':22s} {'bästa val loss':>15s} {'val loss (slut)':>16s} {'bästa val acc':>14s} {'train acc (slut)':>17s}")
    for name, h in results.items():
        print(f"{name:22s} {min(h['val_loss']):>15.3f} {h['val_loss'][-1]:>16.3f} "
              f"{max(h['val_acc']):>14.3f} {h['train_acc'][-1]:>17.3f}")

summarize(results)

Titta främst på **val loss vid slutet**. Utan regularisering stiger den kraftigt när modellen blir allt mer självsäker på sina fel.

Här gör 0,01 och 0,1 nästan ingen skillnad. Först med 1,0 hålls val loss nere och gapet krymper. Det beror på att AdamW krymper vikterna med `lr × weight_decay` per steg. Med `lr=1e-3` och bara cirka 640 steg (2 000 bilder, 40 epoker) hinner en svag decay inte verka. I längre träningar med många fler steg räcker de typiska värdena 0,01–0,1. **Lärdom:** rätt styrka beror på träningens längd och learning rate, så den måste testas mot val (L08). Blir den för stark kommer underanpassning: modellen får inte ha de vikter den behöver.

## 4. Dropout

**Idé:** under träning nollställs varje neuron med sannolikheten `p` (till exempel 0,3) i varje steg. Nätverket kan då inte förlita sig på att en viss neuron finns där och tvingas sprida kunskapen. Det liknar att träna många olika delnätverk och ta medelvärdet av dem.

**Viktigt: dropout är bara aktivt under träning.** Vid utvärdering vill vi använda hela nätet, deterministiskt. Därför är växlingen mellan lägena viktig:
- `model.train()` slår på dropout. `dlkit.train_one_epoch` anropar den.
- `model.eval()` slår av dropout. `dlkit.evaluate` och `dlkit.predict` anropar den.

In [ ]:
drop = nn.Dropout(p=0.5)
x = torch.ones(1, 10)

drop.train()
print("train-läge:", drop(x))
drop.eval()
print("eval-läge: ", drop(x))

I train-läge blir de kvarvarande värdena **2,0**, inte 1,0. PyTorch skalar upp dem med $1/(1-p)$ så att *medelvärdet* blir detsamma som i eval-läge. Annars skulle nästa lager få en annan skala på indata vid utvärdering än under träning.

**Var ska dropout sitta?** Efter aktiveringen i dolda lager, men inte i utdatalagret. Typiskt `p` är 0,1–0,5. Högre `p` ger starkare regularisering.

In [ ]:
def make_mlp_dropout(p, hidden=512):
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, hidden), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(hidden, hidden), nn.ReLU(), nn.Dropout(p),
                         nn.Linear(hidden, 10))

for p in [0.2, 0.5]:
    set_seed(42)
    model = make_mlp_dropout(p).to(device)
    results[f"dropout {p}"] = fit(model, train_loader, val_loader, loss_fn,
                                  torch.optim.Adam(model.parameters(), lr=1e-3), device,
                                  epochs=40, verbose=False)
summarize(results)

## 5. Batch normalization

**Idé:** normalisera *aktiveringarna inuti nätet*, inte bara indata (L04). För varje batch beräknas medel och standardavvikelse per neuron, och aktiveringarna standardiseras. Sedan lär sig lagret själv en lämplig skala och förskjutning.

- **Varför?** Varje lager får indata med stabil skala. Träningen blir snabbare och tål högre learning rate. Det har också en liten regulariserande effekt, eftersom statistiken från varje batch är lite brusig.
- **train kontra eval:** under träning används batchens statistik. Vid utvärdering används ett *glidande medelvärde* som samlats in under träningen. Där också är `model.eval()` avgörande.
- **Var?** Vanligtvis `Linear` (eller `Conv`), sedan `BatchNorm`, sedan `ReLU`.
- **Varning:** med mycket små batcher (under cirka 16) blir statistiken för brusig. Då passar *LayerNorm* bättre (används i transformers, L13).

In [ ]:
bn = nn.BatchNorm1d(3)
batch = torch.randn(64, 3) * 5 + 10           # medel ~10, std ~5
bn.train()
out = bn(batch)
print("efter BN (train): mean", out.mean(0).detach().round(decimals=2), "std", out.std(0).detach().round(decimals=2))
print("glidande medel som sparats för eval:", bn.running_mean.round(decimals=2))
# Bara ~1, inte ~10: running_mean = 0.9 * gammalt + 0.1 * batchens medel (momentum=0.1).
# Efter en batch har det bara hunnit 10 % av vägen från startvärdet 0. Efter många batcher närmar det sig 10.
for _ in range(100):
    bn(torch.randn(64, 3) * 5 + 10)
print("efter 100 batcher till:", bn.running_mean.round(decimals=2))

In [ ]:
def make_mlp_bn(hidden=512):
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                         nn.Linear(hidden, 10))

set_seed(42)
model = make_mlp_bn().to(device)
results["batch norm"] = fit(model, train_loader, val_loader, loss_fn,
                            torch.optim.Adam(model.parameters(), lr=1e-3), device,
                            epochs=40, verbose=False)
summarize(results)

Batch norm används i första hand för att göra träningen **stabilare och snabbare**, särskilt i djupa CNN:er (L10). Regulariseringen är en bieffekt. I en liten MLP som den här är effekten blygsam.

## 6. Jämför kurvorna

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for name, h in results.items():
    ax.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], label=name)
ax.set_xlabel("epok"); ax.set_ylabel("val loss"); ax.set_ylim(0.3, 1.5)
ax.legend(); ax.grid(alpha=0.3); ax.set_title("Val loss med olika regularisering (2 000 träningsbilder)")
plt.show()

## 7. Vad händer om … vi glömmer `model.eval()`?

**Gissa först:** vi utvärderar en dropout-modell men "glömmer" att slå om till eval-läge. Vad händer med accuracy, och blir resultatet detsamma varje gång?

In [ ]:
set_seed(42)
model = make_mlp_dropout(0.5).to(device)
fit(model, train_loader, val_loader, loss_fn, torch.optim.Adam(model.parameters(), lr=1e-3),
    device, epochs=15, verbose=False)

print("rätt (eval-läge):", round(evaluate(model, val_loader, loss_fn, device)[1], 4))

@torch.no_grad()
def evaluate_forgot_eval(model, loader):
    model.train()          # FEL: dropout är fortfarande på
    correct = 0
    for X, y in loader:
        correct += (model(X.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(loader.dataset)

for i in range(3):
    print(f"fel (train-läge), körning {i+1}:", round(evaluate_forgot_eval(model, val_loader), 4))

Resultatet blir sämre *och* olika varje gång. Om dina utvärderingar inte går att upprepa: kontrollera att `model.eval()` anropas.

## 8. Övningar

### Övning 1 – dropout för hand
Skriv `my_dropout(x, p)` som gör samma sak som `nn.Dropout` i train-läge: nollställ med sannolikheten `p` och skala upp resten med $1/(1-p)$.

In [ ]:
def my_dropout(x, p):
    # TODO: använd torch.rand_like(x) för att skapa en slumpmask
    ...

In [ ]:
set_seed(0)
out = my_dropout(torch.ones(100_000), p=0.3)
assert torch.allclose(out[out > 0], torch.tensor(1 / 0.7)), "kvarvarande värden ska vara 1/(1-p)"
assert abs((out == 0).float().mean().item() - 0.3) < 0.01, "andelen nollor ska vara ca p"
assert abs(out.mean().item() - 1.0) < 0.01, "medelvärdet ska bevaras"
print("Rätt!")

### Övning 2 – kombinera
Kombinera **dropout, weight decay (AdamW) och early stopping** på datamängden med 2 000 bilder. Målet är en lägre *bästa val loss* än utan regularisering, och en bästa val accuracy på minst 0,81.

In [ ]:
set_seed(42)
# TODO: modell, optimerare, fit med patience
h_combo = None

In [ ]:
assert min(h_combo["val_loss"]) < min(h_base["val_loss"]), (min(h_combo["val_loss"]), min(h_base["val_loss"]))
assert max(h_combo["val_acc"]) >= 0.81, max(h_combo["val_acc"])
print("Rätt!", round(min(h_combo["val_loss"]), 3), round(max(h_combo["val_acc"]), 3))

### Övning 3 – mer data är den bästa regulariseringen
Träna den **oregulariserade** modellen på alla 50 000 bilder (`train_subset=None`) i 10 epoker. Jämför gapet mellan train och val accuracy med gapet för 2 000 bilder. Vad drar du för slutsats?

In [ ]:
# TODO

*Din slutsats:*

## När använder jag detta?
| Teknik | Använd när | Typiska värden |
|---|---|---|
| **Early stopping** | i princip alltid | `patience` 5–10 |
| **Weight decay** (AdamW) | som standard, och starkare vid överanpassning | 0,01–0,1 i normala träningar. Testa högre vid korta träningar. |
| **Dropout** | stora dense-lager (MLP, klassificeringshuvud) som överanpassar | `p` 0,1–0,5 |
| **Batch norm** | djupa nät, särskilt CNN, för stabilare och snabbare träning | efter Linear/Conv, före ReLU |
| **Mer data och augmentation** | alltid det bästa om det går (L10) | – |

**Arbetsordning:** se först till att modellen *kan* överanpassa, så att kapaciteten räcker. Lägg sedan till regularisering tills val slutar förbättras.

## Sammanfattning
- Regularisering minskar gapet mellan train och val genom att begränsa vad modellen kan lära sig utantill.
- Early stopping behåller den bästa epoken. Weight decay krymper vikterna. Dropout slår av slumpmässiga neuroner.
- Dropout och batch norm beter sig olika i `train()` och `eval()`, så glöm inte att växla.
- `dlkit.fit` har nu `patience=`.

**Nästa lektion:** hyperparametersökning och learning rate schedules.